# MUSCAT Instrument Walkthrough

This notebook demonstrates how to use the AtLAST Sensitivity Calculator with the MUSCAT (Terahertz IFU with Universal Nanotechnology) instrument. MUSCAT is a wideband integral field unit designed for the ASTE telescope.

## MUSCAT Specifications

- **Frequency Range**: 250-300 GHz
- **Type**: Kinetic Inductance Detector 
- **Purpose**: Continuum observations
- **Instrument Webpage**: [MUSCAT webpage](https://muscat-docs.astro.cf.ac.uk/)
- **Instrument Reference**: [Tapia et al. 2020](https://arxiv.org/pdf/2012.05126)

This walkthrough will guide you through:
1. Setting up the calculator with MUSCAT parameters
2. Demonstration of the typical calculator usage
3. Ensuring correct instrument selection
4. Working through the sensitivity equations

## Setup and Imports

First, we import the necessary modules and initialise the calculator using the CalculatorFactory.

In [1]:
# Import utilities
import numpy as np
import astropy.units as u
from astropy import constants

# Import Sensitivity Calculator
from atlast_sc.factory import CalculatorFactory
from atlast_sc.derived_groups import noise_temperature

# Import Plotting routines
import matplotlib.pyplot as plt
# Set up plotting style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## Quick Demonstration of the Calculator Usage

Here we show a quick demonstration of how the typical user will interact with the calculator in the MUSCAT parameter space.

### Initialise the Calculator with MUSCAT Parameters

We'll initialise the calculator with observation parameters that fall within MUSCAT's frequency range and spectral resolution capabilities. MUSCAT has a flexible resolving power that can be varied between 20 and 500. This equates to per channel bandwidths of between 0.18 GHz and 18 GHz.

By setting MUSCAT appropriate observing frequency and spectral channel bandwidth, the AtLAST sensitivity calculator will automatically select the appropriate instrument based on the observing frequency and single channel bandwidth.

In [2]:
# Create calculator instance using CalculatorFactory
calculator = CalculatorFactory().calculator

# Set observation parameters for MUSCAT
calculator.user_input.obs_freq = 275.0 * u.GHz  # MUSCAT's central frequency
calculator.user_input.weather = 25.0  # percentile water vapour (see documentation for conversion to mm)
calculator.user_input.bandwidth = 50 * u.GHz  # MUSCAT's bandwidth 
calculator.user_input.t_int = 30.0 * u.min  # on source time only, no calibration time added

Instrument has been changed from Default to Sepia.
Instrument has been changed from Sepia to Muscat.


The output above shows that the code first selects SEPIA based on the given observing frequency and then changes to MUSCAT once the bandwidth is increased as MUSCAT is more suitable instrument for these parameters. To show the full list of parameters set for the initial calculations, we can ask the calculator to 'show' the full list of inputs available. Those not explicitly set above are passed their default settings.

In [ ]:
print(f"Calculator initialised with the following parameters:")
calculator.user_input.show()

Calculator initialised with the following parameters:
bandwidth: 0.5 GHz
elevation: 45.0 deg
n_pol: 2.0
obs_freq: 230.0 GHz
sensitivity: 3.0 mJy
t_int: 30.0 min
weather: 25.0


### Calculating the Sensitivity or Observing Time for an Observation

Given these inputs we can then calculate the sensitivity reached in a 30 minute observation.

In [23]:
sens = calculator.calculate_sensitivity()
print(f'RMS sensitivity: {sens}')

RMS sensitivity: 5.105936115340478 uJy


Alternatively we can calculate the on source observing time to reach a given sensitivity.

In [5]:
calculator.user_input.sensitivity = 20*u.uJy
obs_time = calculator.calculate_t_integration()
print(f'On source observing time required: {obs_time}')

On source observing time required: 1.9552937710453655 min


This is all that is needed for the typical user. The following steps of this notebook demonstrate the validation of the instrument and then the details of the sensitivity calculation for MUSCAT.

## Ensuring Correct Instrument Selection

The calculator provides validation to ensure that your observing parameters (frequency and bandwidth) are compatible with the chosen instrument. This validation ensures your calculations are internally consistent with your expectations. If the parameters don't match, it raises an `InstrumentNotApplicableException` and suggests ways of setting a valid calculation.

### Why This Validation is Important

This validation prevents you from accidentally using an instrument with parameters outside its valid ranges, which would lead to incorrect sensitivity calculations. The error message helps you either:
- Adjust your parameters to fit the desired instrument, or
- Choose an instrument that matches your observing requirements

### Checking the Selected Instrument

Let's verify which instrument was automatically selected for our observation frequency:

In [6]:
# Check which instrument was automatically selected
print(f"Automatically selected instrument: {calculator.chosen_instrument}")

Automatically selected instrument: Muscat


### Demonstrating Instrument Validation

Let's demonstrate how the instrument validation works. We'll first try to set MUSCAT with incompatible parameters to see the validation in action, then show the correct approach.

In [11]:
# First, let's temporarily set incompatible parameters (bandwidth too small for MUSCAT)
calculator.user_input.bandwidth = 0.01 * u.GHz  # outside MUSCAT's range (min 0.18 GHz)

# This will raise an exception because our bandwidth doesn't fit MUSCAT's ranges
try:
    calculator.chosen_instrument = 'MUSCAT'
    print("Instrument set successfully")
except Exception as e:
    print(f"Cannot set to MUSCAT: {type(e).__name__}")
    print(f"Reason: {str(e)}")

Instrument has been changed from Muscat to Sepia.
Cannot set to MUSCAT: InstrumentNotApplicableException
Reason: Specified observing frequency and/or bandwidth values do not correspond to the chosen instrument 'Muscat' ranges. Change the observing frequency and/or bandwidth values to use this instrument or choose another instrument. The set of parameters provided corresponds to other instrument/s, e.g. 'Sepia'. To check the frequency and bandwidth ranges of the installed instruments type calculator.list_instruments().


Knowing that the setup is wrong, the above error statement is expected. Now lets reset the observing bandwidth to something that is achievable with MUSCAT, and then show that that change has been applied.

In [12]:
# Reset to compatible bandwidth
calculator.user_input.bandwidth = 50.0 * u.GHz  # back within MUSCAT's range
calculator.user_input.show()
print(f"Calculations will now be performed using: {calculator.chosen_instrument}")

Instrument has been changed from Sepia to Muscat.
bandwidth: 50.0 GHz
elevation: 45.0 deg
n_pol: 2.0
obs_freq: 275.0 GHz
sensitivity: 20.0 uJy
t_int: 30.0 min
weather: 25.0
Calculations will now be performed using: Muscat


## System Temperature Calculation

As a KID instrument, the sensitivity is calculated slightly differently to the coherent receiver instruments that the sensitivity calculator was originally designed for as Poisson noise and quasiparticle recombination noise are important in addition to the wave noise, as described in detail in this [note](https://github.com/ukatc/AtLAST_sensitivity_calculator/wiki/Sensitivity-Calculation-for-a-Single%E2%80%90mode-KID-based-Instrument). By considering the resultant sensitivity equivalent to the sensitivity for a heterodyne instrument and re-arranging the equations, we find that we can incorporate this instrument into our sensitivity calculation by determining an equivalent system temperature that is dependent on the Noise Equivalent Power (NEP) as follows:

\begin{equation}
T_{sys} = \frac{\mathrm{NEP}}{k\,\eta_\mathrm{chip,co}\,\eta_\mathrm{eff}\,\mathfrak{t}\,\sqrt{2n_\mathrm{pol}\,\Delta\nu} } \nonumber
\end{equation}

where

* $\eta_\mathrm{eff}$ is the forward efficiency
* $\eta_\mathrm{chip,co}$ is the combination of the chip optical efficiency and cold optics optical efficiency
* $\mathfrak{t}$ is the atmospheric transmittance, defined as $\mathfrak{t} = \textrm{exp}^{(-\tau_{atm})}$

The Noise Equivalent Power is the square root of the sum of the Poisson noise, bunching (wave) noise and quasiparticle recombination noise and calculated as:

\begin{equation}
    \mathrm{NEP} = \sqrt{2\,P_\mathrm{KID}\,h\,\nu+2\,P_\mathrm{KID}^2/(n_\mathrm{pol}\Delta \nu)+4\,\Delta_\mathrm{g}\,P_\mathrm{KID}/\eta_{pb}} \nonumber
\end{equation}

where

* $\Delta_\mathrm{g}$ is the gap energy of the superconductor
* $\eta_\mathrm{pb}$ is the pair-breaking efficiency
* $P_\mathrm{KID}$ is the power received by the KID.

The power received by the KID ($P_\mathrm{KID}$) is dependent on the power spectral density ($PSD_\mathrm{KID}$), which is the sum of the contributions of the noise sources and calculated as:

\begin{align}
P_\mathrm{KID}(\nu_\mathrm{0}) &= \int^{\nu_\mathrm{max}}_{\nu_\mathrm{min}} PSD_\mathrm{KID}(\nu)\: d\nu \sim PSD_\mathrm{KID}(\nu_\mathrm{0}) \Delta \nu \nonumber \\

PSD_\mathrm{KID}(\nu) &=
k(\eta_\mathrm{chip,co}(1-\eta_\mathrm{eff})\, O(\nu, T_\mathrm{amb})  \nonumber\\
&+ \eta_\mathrm{chip,co}\,\eta_\mathrm{eff}\,T_\mathrm{sky})  \nonumber
\end{align}

where 

* $k$ is the Boltzmann constant
* $T_\mathrm{sky}$ is the Rayleigh-Jeans brightness temperature of the sky calculated from the model grid 
* $T_\mathrm{amb}$ is the ambient temperature.

Here, $O(\nu, T)$ is the conversion from a thermodynamic temperature to a Rayleigh-Jeans temperature

\begin{equation}
    O(\nu, T) = T \frac{h\nu/kT}{\exp(h\nu/kT)-1}.  \nonumber
\end{equation}

In the following, we will step through the calculation of the separate components of the power spectral density and the separate components of the noise equivalent power.

### Power Spectral Density Calculation

The power spectral density is the power per unit frequency incident on the detector. Its calculation occurs within the instrument module. The temperature of the cold optics, the ambient temperature of the telescope and the sky temperature all contribute to the power spectral density. The separate components are not extractable by the user and so we recalculate them here in the same manner as in the calculator in order to help with the validation of the instrument implementation. The separate components are given as
\begin{align}
PSD_\mathrm{amb}(\nu) &= k\,\eta_\mathrm{chip,co}(1-\eta_\mathrm{eff})\, O(\nu, T_\mathrm{amb})  \nonumber\\
PSD_\mathrm{sky}(\nu) &= k\,\eta_\mathrm{chip,co}\,\eta_\mathrm{eff}\,T_\mathrm{sky}  \nonumber
\end{align}

In [15]:
# expose the MUSCAT instrument module in order to pull out the parameters used in the calculations
MUSCAT_instrument = calculator._param_setup.chosen_instrument

# read in the constants necessary for the following equations
eta_chip_co = MUSCAT_instrument.eta_chip_co
delta_g = MUSCAT_instrument.delta_g
eta_pb = MUSCAT_instrument.eta_pb
eta_eff = calculator.telescope_and_environment.eta_eff
T_amb = calculator.telescope_and_environment.T_amb

# ensure the frequencies have a unit
frange = calculator.user_input.obs_freq

# read in the sky temperature
t_sky_contribution = calculator.derived_parameters.T_sky

# calculate the power spectral density components for the ambient temperature and sky temperature
# note here that the ambient temperature needs to be converted to a noise temperatures, but
# the sky temperature is already a noise temperature
# note also that the sky temperature includes the cosmic microwave background
psdkid_amb = (eta_chip_co * (1 - eta_eff) * constants.k_B * \
        noise_temperature(T_amb, frange)).to(u.W/u.Hz)
psdkid_sky = (eta_chip_co * eta_eff * constants.k_B * t_sky_contribution).to(u.W/u.Hz)

# calculate the total power spectral density
psdkid = psdkid_amb + psdkid_sky

# print output
print(f'PSD ambient term: {psdkid_amb}')
print(f'PSD sky term: {psdkid_sky}')
print(f'PSD total: {psdkid}')

PSD ambient term: 9.093465135204314e-23 W / Hz
PSD sky term: 8.668487020003244e-23 W / Hz
PSD total: 1.7761952155207558e-22 W / Hz


### Noise Equivalent Power Calculation

The Noise Equivalent Power of a KID is made up of the Poisson noise, wave noise and quasi-particle recombination noise. As with the PSD, these components are calculated within the instrument module and are not extractable by the user and so we recalculate them here to help with the instrument validation. The separate components are calculated as
\begin{align}
    \mathrm{NEP}_\mathrm{Poisson}^2 &= 2\,P_\mathrm{KID}\,h\,\nu \nonumber\\
    \mathrm{NEP}_\mathrm{wave}^2 &= 2\,P_\mathrm{KID}^2/(n_\mathrm{pol}\Delta \nu) \nonumber\\
    \mathrm{NEP}_\mathrm{qpr}^2 &= 4\,\Delta_\mathrm{g}\,P_\mathrm{KID}/\eta_{pb} \nonumber
\end{align}


In [22]:
# Set-up the range of bandwidths
brange = calculator.user_input.bandwidth

# Convert the power spectral density to the power received by the KID
pkid = (psdkid * brange).to(u.W)

# Calculate the separate contributions to the square of the Noise Equivalent Power
nep2poisson = (2 * pkid * constants.h * frange).to(u.Hz**(-1)*u.W**2)
nep2wave = (2 * pkid**2 / (calculator.user_input.n_pol * brange)).to(u.Hz**(-1)*u.W**2)
nep2quasiparticle = (4 * delta_g * pkid / eta_pb).to(u.Hz**(-1)*u.W**2)

# Calculate the total Noise Equivalent Power
nep = np.sqrt(nep2poisson + nep2wave + nep2quasiparticle)

# print output
print(f'NEP^2 Poisson term: {nep2poisson}')
print(f'NEP^2 Wave term: {nep2wave}')
print(f'NEP^2 Quasi-particle recombination term: {nep2quasiparticle}')
print(f'NEP total: {nep}')

NEP^2 Poisson term: 3.236528376987096e-33 W2 / Hz
NEP^2 Wave term: 1.577434721819412e-33 W2 / Hz
NEP^2 Quasi-particle recombination term: 2.8457784717299486e-33 W2 / Hz
NEP total: 8.75199495574378e-17 W / Hz(1/2)


## System temperature, SEFD and Sensitivity

[The System Equivalent Flux Density (SEFD)](https://atlast-sensitivity-calculator.readthedocs.io/en/development/calculator_info/sensitivity.html) characterises the sensitivity of the telescope system. The system temperature derived in the instrument module is used to calculate the overall SEFD for the telescope, which is then used to calculate the sensitivity and provide our final result.

In [19]:
print(f'System temperature: {calculator.derived_parameters.T_sys}')
print(f'SEFD: {calculator.derived_parameters.sefd.to("Jy")}')
print(f'Sensitivity: {calculator.calculated_sensitivity}')

System temperature: 31.547222344177957 K
SEFD: 67.81828824147475 Jy
Sensitivity: 5.105936115340478 uJy


## Summary

This notebook has demonstrated the key capabilities of the AtLAST Sensitivity Calculator for the MUSCAT instrument:

1. **Sensitivity / Observing Time Calculation**: Calculated the sensitivity for a given observing time and vice versa
2. **Instrument Selection**: Verified automatic selection and demonstrated explicit instrument setting
3. **Sensitivity Equations**: Worked through the sensitivity equations for MUSCAT to show where are result came from